In [8]:
from data_frame.aggregation.aggregator import Aggregator
from data_frame.spark_utils import get_spark
from pyspark.sql import functions as F

In [9]:
spark = get_spark(app_name="Grouping and Aggregations")

In [10]:
# Create sales data
sales_data = [
    ("North", "Electronics", 1000, 2),
    ("North", "Electronics", 1500, 3),
    ("South", "Electronics", 2000, 4),
    ("North", "Clothing", 800, 5),
    ("South", "Clothing", 1200, 3),
    ("East", "Electronics", 1800, 2),
    ("East", "Clothing", 900, 4)
]
df_sales = spark.createDataFrame(sales_data, 
                                 ["region", "category", "revenue", "quantity"])

In [11]:
# Basic aggregations
agg_dict = {
    "revenue": ["sum", "avg", "count"],
    "quantity": ["sum", "avg"]
}

result = Aggregator.basic_aggregations(df_sales, ["region"], agg_dict)
print("Aggregations by region:")
result.show()

Aggregations by region:
+------+-----------+-----------+-------------+------------+------------------+
|region|revenue_sum|revenue_avg|revenue_count|quantity_sum|      quantity_avg|
+------+-----------+-----------+-------------+------------+------------------+
| North|       3300|     1100.0|            3|          10|3.3333333333333335|
| South|       3200|     1600.0|            2|           7|               3.5|
|  East|       2700|     1350.0|            2|           6|               3.0|
+------+-----------+-----------+-------------+------------+------------------+



In [12]:

"""
## 2. Multiple Aggregations
"""
metrics = {
    "sum": ["revenue", "quantity"],
    "avg": ["revenue", "quantity"],
    "max": ["revenue"]
}

result_multi = Aggregator.multiple_aggregations(df_sales, ["region", "category"], metrics)
print("Multiple aggregations:")
result_multi.show()

Multiple aggregations:
+------+-----------+-----------+------------+-----------+------------+-----------+
|region|   category|revenue_sum|quantity_sum|revenue_avg|quantity_avg|revenue_max|
+------+-----------+-----------+------------+-----------+------------+-----------+
| North|Electronics|       2500|           5|     1250.0|         2.5|       1500|
| South|Electronics|       2000|           4|     2000.0|         4.0|       2000|
| North|   Clothing|        800|           5|      800.0|         5.0|        800|
| South|   Clothing|       1200|           3|     1200.0|         3.0|       1200|
|  East|Electronics|       1800|           2|     1800.0|         2.0|       1800|
|  East|   Clothing|        900|           4|      900.0|         4.0|        900|
+------+-----------+-----------+------------+-----------+------------+-----------+



In [13]:
"""
## 3. Conditional Aggregations
"""
conditions = {
    "high_revenue": F.col("revenue") > 1500,
    "medium_revenue": (F.col("revenue") >= 1000) & (F.col("revenue") <= 1500),
    "low_revenue": F.col("revenue") < 1000
}

conditional_result = Aggregator.conditional_aggregations(
    df_sales, ["region"], conditions, "revenue", "sum"
)
print("Conditional aggregations:")
conditional_result.show()

Conditional aggregations:
+------+--------------------+----------------------+-------------------+
|region|revenue_high_revenue|revenue_medium_revenue|revenue_low_revenue|
+------+--------------------+----------------------+-------------------+
| North|                   0|                  2500|                800|
| South|                2000|                  1200|                  0|
|  East|                1800|                     0|                900|
+------+--------------------+----------------------+-------------------+

